In [2]:
#FINAL PROGRAMA GRADIO
import gradio as gr
import pandas as pd
from transformers import pipeline
import requests
import re
from geopy.geocoders import Nominatim
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your CSV data
df = pd.read_csv('landslide_events.csv')

# Initialize the summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0)

# Function to generate the summary
def generate_summary(location):
    relevant_events = df[df['location_description'].notna() & df['location_description'].str.contains(location, case=False)]
    
    if relevant_events.empty:
        return "No past landslide events found for this location."
    
    descriptions = relevant_events['event_description'].tolist()
    full_text = " ".join(descriptions)
    
    if len(full_text.split()) < 50:
        return "The information about past events is too short for a summary. Here are the details:\n" + full_text
    
    input_length = len(full_text.split())
    adjusted_max_length = min(15, int(0.6 * input_length))

    max_chunk_size = 1024
    if len(full_text) > max_chunk_size:
        text_chunks = [full_text[i:i + max_chunk_size] for i in range(0, len(full_text), max_chunk_size)]
        summaries = []
        for chunk in text_chunks:
            summary_chunk = summarizer(chunk, max_length=adjusted_max_length, min_length=10, do_sample=False)
            summaries.append(summary_chunk[0]['summary_text'])
    else:
        summary_chunk = summarizer(full_text, max_length=adjusted_max_length, min_length=10, do_sample=False)
        summaries = [summary_chunk[0]['summary_text']]
    
    # Convert summaries into a point-wise format
    pointwise_summary = "\n".join([f"- {summary.strip()}" for summary in summaries])
    return pointwise_summary if pointwise_summary else "No significant points found."




# Function to generate the summary
def generate_summary_(location):
    relevant_events = df[df['location_description'].notna() & df['location_description'].str.contains(location, case=False)]
    
    if relevant_events.empty:
        return "No past landslide events found for this location."
    
    descriptions = relevant_events['event_description'].tolist()
    full_text = " ".join(descriptions)
    
    if len(full_text.split()) < 50:
        return "The information about past events is too short for a summary. Here are the details:\n" + full_text
    
    input_length = len(full_text.split())
    adjusted_max_length = min(20, int(0.8 * input_length))
    
    max_chunk_size = 1024
    if len(full_text) > max_chunk_size:
        text_chunks = [full_text[i:i + max_chunk_size] for i in range(0, len(full_text), max_chunk_size)]
        summary = ""
        for chunk in text_chunks:
            summary_chunk = summarizer(chunk, max_length=adjusted_max_length, min_length=30, do_sample=False)
            summary += summary_chunk[0]['summary_text'] + " "
        return summary.strip()
    else:
        summary = summarizer(full_text, max_length=adjusted_max_length, min_length=30, do_sample=False)
        return summary[0]['summary_text']

# Function to generate detailed reports
def generate_detailed_report(location):
    relevant_events = df[df['location_description'].notna() & df['location_description'].str.contains(location, case=False)]
    
    if relevant_events.empty:
        return "No past landslide events found for this location."

    # Generate detailed event reports
    details = ""
    for index, row in relevant_events.iterrows():
        details += f"<h3>{row['event_title']}</h3>"
        details += f"<p><b>AI Generated Description</b><br> {row['event_description']}</p>"
        details += f"<p><b>Date of incident:</b> {row['event_date']}</p>"
        details += f"<p><b>Location:</b>{row['location_description']}</p><hr>"
        
    return details
# Function to generate charts for a specific city
def generate_visual_info(city_name):
    df_city = df[df['location_description'].str.contains(city_name, case=False, na=False)]

    if df_city.empty:
        return f"No data available for {city_name}."

    # Set a modern style for the plots
    sns.set(style="whitegrid")

    # Plot the frequency of landslide triggers
    plt.figure(figsize=(10, 6))
    colors = sns.color_palette("coolwarm", len(df_city['landslide_trigger'].unique()))
    trigger_plot = sns.countplot(data=df_city, x='landslide_trigger', palette=colors)
    plt.title(f'Frequency of Landslide Triggers in {city_name}', fontsize=16, weight='bold', color='#072b5b')
    plt.xlabel('Landslide Trigger', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Plot the distribution of landslide sizes
    plt.figure(figsize=(8, 8))
    colors = sns.color_palette("Set2")
    df_city['landslide_size'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=colors, startangle=90, wedgeprops={'edgecolor': 'black'})
    plt.title(f'Distribution of Landslide Sizes in {city_name}', fontsize=16, weight='bold', color='#072b5b')
    plt.tight_layout()
    plt.show()
# Function for main page
def process_input(short_url, weather_input, time_input, city, show_details):
    # Simulate extracting location, weather, and elevation details
    # This would be replaced with the actual logic you had before
    # Expand the shortened URL
    response = requests.get(short_url)
    expanded_url = response.url

    try:
        # Extract coordinates from the URL
        if '/search/' in expanded_url:
            split_url = expanded_url.split('/search/')[1].split('?')[0]
            latitude, longitude = split_url.split(',')
        else:
            lat_lng_pattern = r'@(-?\d+\.\d+),(-?\d+\.\d+)'
            match = re.search(lat_lng_pattern, expanded_url)
            if match:
                latitude = match.group(1)
                longitude = match.group(2)
            else:
                raise ValueError("No coordinates found in the URL")

        # Correct the longitude format if necessary
        if (longitude.startswith('+') and longitude[1] == '-') or (longitude.startswith('-') and longitude[1] == '+'):
            longitude = longitude.replace('+', '', 1)

        # Get the location
        geolocator = Nominatim(user_agent="gradio_app")
        location = geolocator.geocode(f"{latitude},{longitude}", timeout=10)
        location_str = location.address if location else "No location found"

        # Get elevation
        elevation_url = f'https://api.open-elevation.com/api/v1/lookup?locations={latitude},{longitude}'
        response = requests.get(elevation_url)
        elevation = response.json()['results'][0]['elevation']
        elevation_str = f"{elevation} meters"

        # Weather input processing
        weather_output = ""
        if weather_input == "1":
            weather_output = "It's raining lightly."
        elif weather_input == "2":
            weather_output = f"It's raining heavily for {time_input} days!"
        elif weather_input == "3":
            weather_output = "It's cloudy."

        # Extracting weather details from Google
        weather_url = "https://www.google.com/search?q=weather+" + city
        html = requests.get(weather_url).content
        soup = BeautifulSoup(html, 'html.parser')

        temp_div = soup.find('div', attrs={'class': 'BNeawe iBp4i AP7Wnd'})
        temp = temp_div.text if temp_div else "Temperature not found"

        str_div = soup.find('div', attrs={'class': 'BNeawe tAd8D AP7Wnd'})
        if str_div:
            data = str_div.text.split('\n')
            time = data[0]
            sky = data[1] if len(data) >= 2 else "Sky description not found"
        else:
            time = sky = "Not found"

        # Summarize past landslide events for the city
        summary = generate_summary(city)
    
        # Generate the HTML output to be displayed as plain text
        html_output = f"""
        <h2>Location: {location_str}</h2>
        <p>Elevation: {elevation_str}</p>
        <p>Temperature: {temp}</p>
        <p>Time: {time}</p>
        <p>Sky Description: {sky}</p>
        """
        # Generate detailed event reports if requested
        detailed_report = ""
        if show_details:
            report_html = """
            <h5>GenAI Suggested Reports</h5>
            """
            detailed_report = generate_detailed_report(city)
            
            # Append the generated report to the report_html
            detailed_report = report_html + detailed_report
        return html_output, summary, detailed_report
    except Exception as e:
        return str(e), "", "", "", "", ""

    return html_output, summary, detailed_report

custom_css = """
    @import url('https://fonts.googleapis.com/css2?family=Krona+One&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Afacad+Flux:wght@100..1000&family=Archivo+Black&family=Lexend:wght@100..900&family=Marcellus&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Afacad+Flux:wght@100..1000&family=Archivo+Black&family=DM+Serif+Display:ital@0;1&family=Lexend:wght@100..900&family=Marcellus&display=swap');

    body {
        background-color: #e3f2ff;
    }
    h1 {
        font-family: 'Krona One', sans-serif;
        text-align: center;
        font-size: 80px;
        color: #000000;
    }
    .marcellus-regular {
      font-family: "Marcellus", serif;
      font-weight: 400;
      font-style: normal;
    }

    .archivo-black-regular {
      font-family: "Archivo Black", sans-serif;
      font-weight: 400;
      font-style: normal;
    }
    .lexend-nika {
      font-family: "Lexend", sans-serif;
      font-optical-sizing: auto;
      font-weight: <weight>;
      font-style: normal;
    }
    .dm-serif-display-regular {
      font-family: "DM Serif Display", serif;
      font-weight: 400;
      font-style: normal;
    }
    
    .dm-serif-display-regular-italic {
      font-family: "DM Serif Display", serif;
      font-weight: 400;
      font-style: italic;
    }

    h4
    {
        font-family: "Lexend", sans-serif;
        text-align: center;
        color: ##212121;
    }
    h5
    {
        font-family: "DM Serif Display", serif;
        text-align: center;
        color: ##212121;
        font-size: 40px;
    }
    .gradio-container {
        background-color: #e3f2ff; 
        border-radius: 10px;
        box-shadow: 0 4px 20px rgba(0,0,0,0);
        padding: 20px;
    }
    .gradio-button {
        background-color: #64b5f6;
        color: white;
        border-radius: 5px;
        padding: 10px;
        font-weight: bold;
    }
    .gradio-button:hover {
        background-color: #42a5f5;
    }
    .gradio-textbox {
        border: 2px solid #64b5f6;
        border-radius: 5px;
        padding: 10px;
    }
    #output-area {
        display: none;
        opacity: 0;
        transition: opacity 0.5s ease-in-out;
        background-color: #e3f2ff;
        padding: 10px;
        border-radius: 5px;
    }
    #output-area.show {
        display: block;
        opacity: 1;
    }
"""
# Gradio function to display visual information using the city name
def show_visual_info(city_name):
    df_city = df[df['location_description'].str.contains(city_name, case=False, na=False)]

    if df_city.empty:
        return f"No data available for {city_name}."

    # Set a modern style for the plots
    plt.figure(figsize=(10, 6))
    colors = sns.color_palette("coolwarm", len(df_city['landslide_trigger'].unique()))
    
    # Plot the frequency of landslide triggers
    plt.subplot(2, 1, 1)  # Create a subplot
    trigger_plot = sns.countplot(data=df_city, x='landslide_trigger', palette=colors)
    plt.title(f'Frequency of Landslide Triggers in {city_name}', fontsize=16, weight='bold', color='#072b5b')
    plt.xlabel('Landslide Trigger', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45)
    
    # Plot the distribution of landslide sizes as a pie chart with a legend
    plt.subplot(2, 1, 2)  # Create another subplot
    sizes = df_city['landslide_size'].value_counts()
    colors = sns.color_palette("Set2")

    # Create the pie chart without labels, using a legend
    plt.pie(sizes, colors=colors, startangle=90, wedgeprops={'edgecolor': 'black'})
    plt.title(f'Distribution of Landslide Sizes in {city_name}', fontsize=16, weight='bold', color='#072b5b')

    # Adding a legend
    plt.legend(sizes.index + [f'{size/ sizes.sum() * 100:.1f}%' for size in sizes], title="Landslide Sizes", loc='center left', bbox_to_anchor=(1, 0, 0.5, 1))

    plt.tight_layout()  # Adjust layout to prevent overlap
    return plt

# Gradio function to display visual information using the city name
def show_visual_info_pv(city_name):
    df_city = df[df['location_description'].str.contains(city_name, case=False, na=False)]

    if df_city.empty:
        return f"No data available for {city_name}."

    # Set a modern style for the plots
    plt.figure(figsize=(10, 6))
    colors = sns.color_palette("coolwarm", len(df_city['landslide_trigger'].unique()))
    
    # Plot the frequency of landslide triggers
    plt.subplot(2, 1, 1)  # Create a subplot
    trigger_plot = sns.countplot(data=df_city, x='landslide_trigger', palette=colors)
    plt.title(f'Frequency of Landslide Triggers in {city_name}', fontsize=16, weight='bold', color='#072b5b')
    plt.xlabel('Landslide Trigger', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45)


    # Plot the distribution of landslide sizes as a pie chart
    plt.subplot(2, 1, 2)  # Create another subplot
    colors = sns.color_palette("Set2")
    df_city['landslide_size'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=colors, startangle=90, wedgeprops={'edgecolor': 'black'})
    plt.title(f'Distribution of Landslide Sizes in {city_name}', fontsize=16, weight='bold', color='#072b5b')
    
    plt.tight_layout()  # Adjust layout to prevent overlap
    return plt
def show_contact_info(city_name):
    url = f"https://www.google.com/search?q=emergency+contact+numbers+for+{city_name}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    html = requests.get(url, headers=headers).content
    soup = BeautifulSoup(html, 'html.parser')
    
    # Finding the specific table that contains the emergency numbers
    table = soup.find('table')  # Find the first table in the search results
    if table:
        contacts = {}
        rows = table.find_all('tr')  # Get all rows in the table
        for row in rows[1:]:  # Skip the header row
            cols = row.find_all('td')  # Get all columns in the row
            if len(cols) >= 2:  # Ensure there are enough columns
                title = cols[0].text.strip()  # Title of the service
                number = cols[1].text.strip()  # Phone number
                contacts[title] = number
    else:
        return f"No emergency contact numbers found for {city_name}."
    html_output = ""
    
    # Iterate over the dictionary and format each key-value pair
    for key, value in contacts.items():
        html_output += f"<p><strong>{key.capitalize()}:</strong> {value}</p>\n"
    
    return html_output
    
# Gradio app interface
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("<h1>TerraAI</h1>")
    gr.Markdown("<h4>Powered By NVIDIA AI Workbench<br>Sowparnika Nair</h4>")
    
    short_url = gr.Textbox(label="🔗 Shortened Google Maps URL", placeholder="Paste your shortened URL here...")
    weather_input = gr.Radio(["Light Rain", "Heavy Rain", "Cloudy"], label="☔ Weather Condition", value="Light Rain")
    time_input = gr.Slider(1, 10, label="📅 How many days has it been raining?", value=1)
    city = gr.Textbox(label="🌆 Enter City Name", placeholder="Type your city here...")

    html_output = gr.HTML(label="Location and Weather Details")
    
    
    summary_output = gr.Markdown(label="Summary of Past Landslide Events")
    show_details = gr.Checkbox(label="Would you like to see detailed reports of each event?", value=False)
    detailed_report_output = gr.HTML(label="Detailed Reports")

    
    submit_btn = gr.Button("🚀 Submit")

    submit_btn.click(
        process_input,
        inputs=[short_url, weather_input, time_input, city, show_details],
        outputs=[html_output, summary_output, detailed_report_output]
    )

    # Graphical output button
    visual_info_button = gr.Button("📊 Show Visual Information")
    
    # Visualizations placeholder (use Plot for more control)
    visual_output = gr.Plot()
    
    # Set up interaction, passing the city input directly to the visual info function
    visual_info_button.click(show_visual_info, inputs=[city], outputs=visual_output)
    
    html_output_ = gr.HTML(label="Emeergency Contacts")
    contact_btn = gr.Button("Emergency Contacts")
    contact_btn.click(show_contact_info, inputs=[city], outputs=html_output_)


demo.launch(share=True)


/home/workbench/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Running on local URL:  http://127.0.0.1:7861
Running on public URL: https://38686ed3d79bbd2dea.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
/tmp/ipykernel_9029/471060178.py:317: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  trigger_plot = sns.countplot(data=df_city, x='landslide_trigger', palette=colors)
Traceback (most recent call last):
  File "/home/workbench/.local/lib/python3.10/site-packages/gradio/queueing.py", line 536, in process_events
    response = await route_utils.call_process_api(
  File "/home/workbench/.local/lib/python3.10/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
  File "/home/workbench/.local/lib/python3.10/site-packages/gradio/blocks.py", line 1945, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
  File "/home/workbench/.local/lib/python3.10